# Lab 15: Haar Bases and Wavelet Transforms

Independent-study notebook with worked solutions.

## Goals

- Compute fast Haar transforms.
- Reconstruct signals.
- Compress by thresholding coefficients.
- Apply a 2D Haar transform to a small image matrix.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Fast Haar transform

In [ ]:
def haar_transform(x):
    x = np.asarray(x, dtype=float).copy()
    coeffs = x.copy()
    length = len(coeffs)
    while length > 1:
        avg = (coeffs[:length:2] + coeffs[1:length:2]) / 2
        diff = (coeffs[:length:2] - coeffs[1:length:2]) / 2
        coeffs[:length//2] = avg
        coeffs[length//2:length] = diff
        length //= 2
    return coeffs

def inverse_haar_transform(c):
    c = np.asarray(c, dtype=float).copy()
    x = c.copy()
    length = 1
    n = len(x)
    while length < n:
        avg = x[:length].copy()
        diff = x[length:2*length].copy()
        x[:2*length:2] = avg + diff
        x[1:2*length:2] = avg - diff
        length *= 2
    return x

### Example: lecture signal

In [ ]:
u = np.array([31, 29, 23, 17, -6, -8, -2, -4])
c = haar_transform(u)
print('Haar coefficients:', c)
print('Reconstruction:', inverse_haar_transform(c))

## Question 1

Compute the Haar transform of $u=(8,6,4,2,1,1,0,0)$.

In [ ]:
u1 = np.array([8,6,4,2,1,1,0,0], dtype=float)
c1 = haar_transform(u1)
print(c1)
print(inverse_haar_transform(c1))

**Answer.** The first coefficient is the global average. Larger-scale coefficients describe block differences, while the final four coefficients describe adjacent-pair differences.

## 2. Compression by thresholding

In [ ]:
u = np.array([2.4, 2.2, 2.15, 2.05, 6.8, 2.8, -1.1, -1.3])
c = haar_transform(u)
for threshold in [0.1, 0.2, 0.5, 1.0]:
    chat = c.copy()
    chat[np.abs(chat) <= threshold] = 0
    uhat = inverse_haar_transform(chat)
    print('threshold =', threshold)
    print('coefficients kept =', np.count_nonzero(chat))
    print('reconstruction =', np.round(uhat, 3))
    print('error =', np.linalg.norm(u-uhat))
    print()

### Visualization

In [ ]:
threshold = 0.5
chat = c.copy()
chat[np.abs(chat) <= threshold] = 0
uhat = inverse_haar_transform(chat)

plt.figure(figsize=(8,4))
plt.plot(u, marker='o', label='original')
plt.plot(uhat, marker='s', label='reconstructed')
plt.title('Haar compression by thresholding')
plt.xlabel('index')
plt.ylabel('value')
plt.legend()
plt.grid(True)
plt.show()

## Question 2

Explain why increasing the threshold usually increases compression but also increases error.

**Answer.** A larger threshold sets more coefficients to zero. This stores fewer nonzero numbers, so compression improves. But discarding more coefficients removes more information, so the reconstruction error usually increases.

## 3. Two-dimensional Haar transform

In [ ]:
def haar2(A):
    A = np.asarray(A, dtype=float)
    C = np.apply_along_axis(haar_transform, 1, A)
    C = np.apply_along_axis(haar_transform, 0, C)
    return C

def inverse_haar2(C):
    A = np.apply_along_axis(inverse_haar_transform, 0, C)
    A = np.apply_along_axis(inverse_haar_transform, 1, A)
    return A

A = np.array([[70,56,61,49],
              [52,46,39,43],
              [63,45,46,54],
              [53,39,40,44]], dtype=float)
C = haar2(A)
print('2D Haar coefficients:')
print(np.round(C,2))
print('Reconstruction check:')
print(np.round(inverse_haar2(C),2))

In [ ]:
C2 = C.copy()
C2[np.abs(C2) < 3] = 0
Ahat = inverse_haar2(C2)
print('Thresholded coefficients:')
print(np.round(C2,2))
print('Reconstructed matrix:')
print(np.round(Ahat,2))
print('error norm =', np.linalg.norm(A-Ahat))

## Question 3

For $A=\begin{bmatrix}10&8\\4&2\end{bmatrix}$, compute the average, vertical, horizontal, and diagonal quantities.

In [ ]:
a,b,c_,d = 10,8,4,2
quantities = [(a+b+c_+d)/4, (a+b-c_-d)/4, (a-b+c_-d)/4, (a-b-c_+d)/4]
print(quantities)

**Answer.** The quantities are $6,3,1,0$.

## 4. Similar practice

Try a signal with constant adjacent pairs, such as $u=(10,10,8,8,2,2,0,0)$. Predict the finest detail coefficients before computing.

In [ ]:
u2 = np.array([10,10,8,8,2,2,0,0], dtype=float)
print(haar_transform(u2))

**Answer.** The finest adjacent-pair detail coefficients are zero because each adjacent pair has equal entries.

## 5. AI companion activity

Ask an AI assistant to compare Haar, Fourier, and SVD compression. Then verify that the answer mentions localization, thresholding, and low-rank approximation.